# HW100 · 1/3 — Build and compile the 120-qubit experiment

**Question.** Can feasible-by-construction states be produced *on a real chip at about 100 qubits*, where penalty / unbalanced / Fourier-LCU methods give almost no feasible samples?

**Setting.** Portfolio with 10 assets, 6 periods, 2 sectors, long/short variables: **n = 120 binary variables**. The interface (per period and sector counts of longs and shorts) is drawn classically; conditioned on it the problem splits into **12 independent blocks of 10 qubits**. All 12 blocks run in **one circuit of 120 qubits** on disjoint regions of the chip. Each block carries its own quadratic objective (the neighbours enter as linear fields).

**Methods (same block QUBO, same shots, same regions).**
`ours` (light p=1), `ours_full` (p=1, full objective and mixer), `ours_p0` (support-F preparation only), `uniform` (Hadamards), and three baselines that all use the **warm start of the Fourier-LCU paper** (RY(2·asin√(k/m)) per qubit): `lcu` (single-basis Fourier-LCU variant, trained), `unbalanced` (−l₁g + l₂g² + exclusivity, trained) and `xy` (XY mixer inside each count group, exclusivity by phase, trained).
The baselines receive the same block decomposition and the same count constraints, which is the setting most favourable to them.

**What this notebook does not need:** the IBM key (it uses the FakeFez snapshot) unless you set `USE_LIVE_BACKEND = True` to select regions with the live calibration. Do that right before running notebook 2, since calibration drifts.

In [ ]:
import os, sys, json, pickle, time, warnings; warnings.filterwarnings('ignore')
root = os.path.abspath('.')
while not os.path.isdir(os.path.join(root, 'negsearch')): root = os.path.dirname(root)      # repository root
os.chdir(root); sys.path.insert(0, root)
import numpy as np
from negsearch import hw100 as H
OUT = 'hardware_circuits/hw100'; RES = 'results/hw100'; os.makedirs(OUT, exist_ok=True); os.makedirs(RES, exist_ok=True)
print('repository root:', root)

In [ ]:
# ---------------- configuration ----------------
USE_LIVE_BACKEND = False          # True: choose regions with the live calibration of BACKEND_NAME (needs the key file)
BACKEND_NAME = 'ibm_fez'
APIKEY = 'apikey_personal.json'   # local file, never committed:  {"apikey": "..."}
INSTANCE = None                   # optional IBM Cloud instance CRN
SEED_OBJ, SEED_IFACE = 5, 11
COMPILE_SEEDS = 4                 # transpiler seeds per block (best ESP kept)
PACK_TRIES = 3000

## 1. Classical part: instance and interface

In [ ]:
inst = H.build_instance(SEED_OBJ, SEED_IFACE)
print('variables:', inst['nv'], '| blocks:', len(inst['blocks']), '| qubits per block:', H.NB_Q)
print('interface (per period: (longs, shorts) of sector 0 and sector 1):')
for t, row in enumerate(inst['iface']): print('  period', t, row)
print('feasible strings per block:', [int(b['mask'].sum()) for b in inst['blocks']], 'of', 2 ** H.NB_Q)
print('log2 of the number of globally feasible strings:', round(sum(np.log2(b['mask'].sum()) for b in inst['blocks']), 1))

## 2. Train the angles (exact simulation of each 10-qubit block; ~1–2 min)

In [ ]:
t0 = time.time()
params = H.train_all(inst, seed=0)
ideal = H.ideal_probs(inst, params)
print(f'trained in {time.time()-t0:.0f}s')

In [ ]:
import pandas as pd
rows = []
for m in H.METHODS:
    mt = [H.metrics(b, d[m]) for b, d in zip(inst['blocks'], ideal)]
    rows.append(dict(method=m, ideal_feasible=np.mean([x['feas'] for x in mt]), ideal_quality=np.mean([x['quality'] for x in mt]),
                     ideal_P_opt=np.mean([x['p_opt'] for x in mt])))
print('IDEAL (noiseless) per-block averages.  quality = 1 - normalised cost among feasible samples (uniform on F gives about 0.5)')
pd.DataFrame(rows).round(3)

## 3. Compile for the chip
Disjoint connected regions of 11 physical qubits (one spare for routing), broken couplers excluded, best regions to the heaviest blocks. Each block is transpiled onto *its own region only* (no cross-block routing) and the blocks are assembled into one circuit with one 120-bit classical register (bit 10·b + k = variable k of block b).

In [ ]:
if USE_LIVE_BACKEND:
    svc, backend = H.connect(APIKEY, BACKEND_NAME, INSTANCE)
else:
    from qiskit_ibm_runtime.fake_provider import FakeFez
    backend = FakeFez()
print('backend:', backend.name, backend.num_qubits, 'qubits')
t0 = time.time()
comp = H.compile_all(inst, params, backend, seeds=COMPILE_SEEDS, tries=PACK_TRIES)
print(f'compiled in {time.time()-t0:.0f}s')
for m in H.METHODS: assert H.check_isa(comp[m]['circuit'], backend), m
print('all circuits use only native gates on real couplers')

In [ ]:
rows = []
for m in H.METHODS:
    st = comp[m]['blocks']
    rows.append(dict(method=m, CZ_total=sum(s['cz'] for s in st), CZ_per_block_median=np.median([s['cz'] for s in st]),
                     depth_max=max(s['depth'] for s in st), ESP_block_median=np.median([s['esp'] for s in st]), ESP_block_min=min(s['esp'] for s in st),
                     qubits_per_circuit=12 * 10))
pd.DataFrame(rows).round(3)

## 4. Pre-registered prediction (global depolarizing model per block)
`output = ESP·ideal + (1−ESP)·uniform` for each block, with ESP the product of (1 − error) over the compiled gates of that block. Measured results from notebook 3 are compared with this table. The model was validated against gate-level noisy simulation up to about 20 qubits (paper, Sec. 9), where it was somewhat pessimistic.

In [ ]:
pred = []
for m in H.METHODS:
    esp = [s['esp'] for s in comp[m]['blocks']]
    P = H.model_probs(ideal, esp, m)
    mt = [H.metrics(b, p) for b, p in zip(inst['blocks'], P)]
    pred.append(dict(method=m, feasible_per_block=np.mean([x['feas'] for x in mt]), quality_feasible=np.mean([x['quality'] for x in mt]),
                     expected_feasible_blocks_per_shot=12 * np.mean([x['feas'] for x in mt])))
pred_df = pd.DataFrame(pred); pred_df.round(3)

## 5. Save circuits and metadata (no credentials involved)

In [ ]:
from qiskit import qpy
for m in H.METHODS:
    with open(f'{OUT}/{m}_120q.qpy', 'wb') as f: qpy.dump(comp[m]['circuit'], f)
    with open(f'{OUT}/{m}_blocks.qpy', 'wb') as f: qpy.dump(comp[m]['subs'], f)
meta = dict(backend=backend.name, live=USE_LIVE_BACKEND, seed_obj=SEED_OBJ, seed_iface=SEED_IFACE, iface=inst['iface'],
            variants=H.VARIANTS, methods=H.METHODS,
            compile={m: [{k: v for k, v in s.items()} for s in comp[m]['blocks']] for m in H.METHODS},
            regions={m: comp[m]['regions'] for m in H.METHODS}, prediction=pred_df.to_dict('records'))
json.dump(meta, open(f'{OUT}/meta.json', 'w'), indent=1, default=float)
pickle.dump(dict(inst=inst, params=params, ideal=ideal), open(f'{OUT}/instance.pkl', 'wb'))
print('saved to', OUT, ':', sorted(os.listdir(OUT)))